In [154]:
import torchvision.models as models
import pandas as pd
import torch
import torch.nn as nn # just get that all up in here. 
import os
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms


from torch import optim

from torch.optim.lr_scheduler import StepLR
from imblearn.over_sampling import SMOTE
from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset, DataLoader

# for toy model, use a SUBSET so we aren't tearing our hair out trying to run this. 
from torch.utils.data import Subset
# same thing that we did for the MLP
from sklearn.metrics import (roc_auc_score, confusion_matrix, 
                             classification_report, fbeta_score)
import numpy as np



In [155]:
# refer to this later when you wnat to instead of fine tuning a network like we have been doing
# want to actually use the chexpert one (bc of course they published it with a python library)
# and then finetune THAT on my dataset. 

# # Install torchxrayvision (has CheXNet pretrained on NIH + CheXpert)
# !pip install torchxrayvision

# import torchxrayvision as xrv
# model = xrv.models.DenseNet(weights="densenet121-res224-all")  # Trained on 14 diseases
# # Replace classifier for binary pneumonia
# model.classifier = nn.Linear(1024, 1)

In [156]:
device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")

In [157]:
class ChestXRayDataset(Dataset):
  def __init__(self, records, image_dir, transform=None):
    """
    records    : a dataframe slice (post train test split)
    image_dir : path to folder containing all .dcm.png files
    transform : torchvision transforms pipeline
    """
    self.records = records.reset_index(drop=True)
    self.image_dir = image_dir
    self.transform = transform

  def __len__(self):
    return len(self.records)

  def __getitem__(self, idx):
    row = self.records.iloc[idx]

    # for image stuff
    img_path = os.path.join(self.image_dir, row['filename'])
    image = Image.open(img_path).convert('RGB') # Make sure to keep this in Black and white, rather than anything else. 
    if self.transform:
      image = self.transform(image)

    # label time
    label = torch.tensor(row["pneumonia_label"], dtype=torch.float32)

    return image, label

In [158]:
# # DF creation. 
# # this is still technically wrong, I want to change from abnormal to pneumonia. There's a lot of other stuff that needs to happen too for that to work.
# def build_master_df_for_pneumonia(projections_path, reports_path):
#     proj = pd.read_csv(projections_path)
#     rep = pd.read_csv(reports_path)
    
#     proj = proj[proj["projection"] == "Frontal"]
#     df = proj.merge(rep[["uid", "Problems"]], on="uid", how="inner")
    
#     def label_pneumonia(row):
#         text = str(row.get("Problems", "")).lower().strip()
#         # Look for explicit pneumonia mentions
#         pneumonia_keywords = ['pneumonia', 'consolidation', 'infiltrate', 'opacity']
        
#         # This is simplified—real CheXNet used NLP extraction from reports
#         # But for your purpose: 1 if pneumonia-related, 0 otherwise
#         if any(keyword in text for keyword in pneumonia_keywords):
#             return 1
#         return 0
    
#     df["pneumonia_label"] = df.apply(label_pneumonia, axis=1)
    
#     # You can drop uid entirely if you want—it's not needed
#     return df[["filename", "pneumonia_label"]]


import re
# USE BETTER VERSION OF THE PNEUMONIA LABEL CREATOR
def simple_negation_check(text, target_terms):
    """Return True if any target term is present and not negated."""
    if not isinstance(text, str):
        return False
    text_lower = text.lower()
    
    if not any(term in text_lower for term in target_terms):
        return False
    
    # Negation patterns (window of up to 5 words)
    neg_patterns = [
        r'\bno\s+(?:\w+\s+){0,5}(?:' + '|'.join(target_terms) + r')\b',
        r'\bwithout\s+(?:\w+\s+){0,5}(?:' + '|'.join(target_terms) + r')\b',
        r'\bfree\s+of\s+(?:\w+\s+){0,5}(?:' + '|'.join(target_terms) + r')\b',
        r'\bnegative\s+for\s+(?:\w+\s+){0,5}(?:' + '|'.join(target_terms) + r')\b',
        r'\bnot\s+(?:\w+\s+){0,5}(?:' + '|'.join(target_terms) + r')\b',
    ]
    for pat in neg_patterns:
        if re.search(pat, text_lower):
            return False
    return True

PNEUMONIA_TERMS = ['pneumonia', 'consolidation', 'infiltrate', 'airspace disease', 'opacity']

def build_master_df_for_pneumonia(projections_path, reports_path):
    proj = pd.read_csv(projections_path)
    rep = pd.read_csv(reports_path)
    
    # Create label column directly on rep
    rep["pneumonia_label"] = rep.apply(
        lambda row: 1 if simple_negation_check(
            f"{row.get('findings','')} {row.get('impression','')}",
            PNEUMONIA_TERMS
        ) else 0,
        axis=1
    )
    
    # Now merge only what's needed
    proj = proj[proj["projection"] == "Frontal"]
    df = proj.merge(rep[['uid', 'pneumonia_label']], on='uid', how='inner')
    
    return df[["filename", "pneumonia_label"]]

In [159]:
def get_dataloaders(projections_path, reports_path, image_dir, test_size=0.2,
                    batch_size=128, image_size=224, num_workers=0, subset=False):
  # need to use 224 for 2 reasons: its what they used in the paper, and
  # its what ImageNet is trained on, so we need to ensure compatability. 

  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  
  # this is wrong now. I need to fix this. 
  df = build_master_df_for_pneumonia(projections_path, reports_path)

  train_df, test_df = train_test_split(
      df,
      test_size=test_size,
      random_state=42,
      stratify=df["pneumonia_label"] # make sure to get a good split. 
  )

  # DOING some standard imageNet transformations
  train_transforms = transforms.Compose([
      transforms.Resize((224, 224)),
      transforms.RandomHorizontalFlip(),
      transforms.RandomRotation(15),
      transforms.ToTensor(),
      transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
  ])

  test_transforms = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
  ])

  train_dataset = ChestXRayDataset(train_df, image_dir, transform=train_transforms)
  test_dataset = ChestXRayDataset(test_df, image_dir, transform=test_transforms)


  # go aheand and just do this here...
  n_neg = (train_df['pneumonia_label'] == 0).sum()
  n_pos = (train_df['pneumonia_label'] == 1).sum()
  pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32).to(device)

  # if we wnat a subset, just create smaller ones, otherwise do nothing. 
  #  just rip this completely out. 
#   subset = True # just do this here super quick. 

#   if subset == True:
#     train_dataset = Subset(train_dataset, range(200))
#     test_dataset = Subset(test_dataset, range(50))


  train_loader = DataLoader(
      train_dataset,
      batch_size=batch_size,
      shuffle=True,
      num_workers=num_workers,
      pin_memory=True if device.type == 'cuda' else False  # speeds up GPU transfer... supposedly. 
  )

  test_loader = DataLoader(
      test_dataset,
      batch_size=batch_size,
      shuffle=False,   # ← never shuffle test
      num_workers=num_workers,
      pin_memory=True if device.type == 'cuda' else False
  )


  # get the df for NLP purposes, will probably find a better way to do it later.
  return train_loader, test_loader, pos_weight

In [160]:
class FinetunedDenseNet(nn.Module):
    def __init__(self):
        super().__init__()
        
        # Load DenseNet-121 with pretrained ImageNet weights
        densenet = models.densenet121(weights="IMAGENET1K_V1")
        
        # don't change these, we will convert to RGB. better for the model. 
        self.features = densenet.features
        
        # Replace classifier for binary pneumonia detection
        # DenseNet features output is (batch, 1024, 7, 7) for 224x224 input
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),   # (batch, 1024, 7, 7) → (batch, 1024, 1, 1) # THIS IS NECESSARY DON"T TRY TO DO STUFF WITHOUT IT. 
            nn.Flatten(),                    # → (batch, 1024)
            nn.Linear(1024, 1)               # → (batch, 1)
        )
    
    def forward(self, x):
        x = self.features(x)      # x shape: (batch, 3, 224, 224) → (batch, 1024, 7, 7)
        x = self.classifier(x)    # → (batch, 1)
        return x

In [161]:
class ProgressiveUnfreezer:
    """ man I should make more docstrings like this, this is nuts. 
    Progressive unfreezing for Densenet121 
    usage:
        unfreezer = ProgresiveUnfreezer(model)

        for epoch in range(num_epochs):
            if freezer.should_update(epochs):
                optimizer = unfreezer.update(epoch, optmiizer)
    """

    def __init__(self, model, strategy="Progressive"):
        self.model = model
        self.strategy = strategy
        self.current_stage = 0
        self.stage_epochs = {0: 3, 1: 5, 2: 5, 3: 10} # odd cob. I don't like hard coding. 

        self._freeze_all()
        self._unfreeze_classifier()

    def _freeze_all(self):
        for param in self.model.parameters():
            param.requires_grad = False

    def _unfreeze_classifier(self):
        for param in self.model.classifier.parameters():
            param.requires_grad = True

    def _unfreeze_denseblock(self, block_num):
        # unfreeze a specific dense block (1-4)
        block_name = f'denseblock{block_num}'
        for name, param in self.model.features.named_parameters():
            if block_name in name:
                param.requires_grad = True

    def _unfreeze_transition(self, trans_num):
        trans_name = f'transition{trans_num}'
        for name, param in self.model.features.named_parameters():
            if trans_name in name:
                param.requires_grad = True

    # ok how exactly does htis function work I no understand good 
    def get_stage_for_epoch(self, epoch):
        "Determin which stage of unfreezing we should be given the epoch"
        cumulative = 0
        for stage, ep in self.stage_epochs.items():
            cumulative += ep
            if epoch < cumulative:
                return stage
            
        return max(self.stage_epochs.keys())
    
    def should_update(self, epoch):
        # check if we need to change unfreezing stage
        new_stage = self.get_stage_for_epoch(epoch)
        return new_stage != self.current_stage
    
    def update(self, epoch, old_optimizer=None):
        new_stage = self.get_stage_for_epoch(epoch)

        if new_stage == self.current_stage:
            return old_optimizer
        
        print("transition to new infreezing stage")

        if new_stage == 0: # only the classifier affected
            pass
            lr = 1e-3
            description = "Classifier only"

        elif new_stage == 1:
            self._unfreeze_denseblock(4)
            lr = 1e-4
            description = "Classifier + Denseblock 4"

        elif new_stage == 2:
            self._unfreeze_denseblock(3)
            self._unfreeze_transition(3)
            lr = 5e-5 # tiny boy
            description = "Classifier and denseblocks 3, 4"
        
        # worth making a functgion called "unfreeze all" just for consistency?
        elif new_stage == 3:
            # everything baby full model time
            for param in self.model.features.parameters():
                param.requires_grad = True
            lr = 1e-5
            description = "Full model fine tuning"

        self.current_stage = new_stage

        # have a hard time trying to care about print statements. 
        trainable = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        total = sum(p.numel() for p in self.model.parameters())
        
        print(f"Stage: {description}")
        print(f"Learning Rate: {lr}")
        print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")
        print(f"{'='*50}\n")


        # now we gotta create a new optimizer with those updated parameters. 
        if old_optimizer is not None:
            optimizer = torch.optim.AdamW(
                filter(lambda p: p.requires_grad, self.model.parameters()), 
                lr=lr, 
                weight_decay=1e-5
            )
            return optimizer
        
        return None # ok

        

In [162]:
# Constants
projections_path = 'IU-xray/indiana_projections.csv'
# this contains the labelling we want for our image CNN
reports_path = 'IU-xray/indiana_reports.csv'
# and of course, the images
image_dir = 'IU-xray/images/images_normalized'

In [ ]:
def train_model(model, train_loader, val_loader, num_epochs=23, device="cuda", pos_weight=0.5):

    model = model.to(device)

    unfreezer = ProgressiveUnfreezer(model)

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()), 
        lr=1e-3, 
        weight_decay=1e-5
    )

    # need to make sure this gets passed in. fantastic. 
    loss_function = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    torch.backends.cudnn.benchmark = True # apparently this is supposed to help
    

    best_val_auc = 0.0 # floor value
    history = {"train_loss": [], "val_loss": [], "val_auc": []}

    for epoch in range(num_epochs):
        # first lets make sure we have all the appropriate layers that we want. 
        if unfreezer.should_update(epoch):
            optimizer = unfreezer.update(epoch, optimizer)

        # training time
        model.train()
        train_loss = 0.0

        for images, labels in train_loader:
            images = images.to(device, non_blocking=True) # non blocking for even more speed
            labels = labels.to(device, non_blocking=True) # ditto


            optimizer.zero_grad(set_to_none=True) # apparently more efficient than zero_Grad()
            outputs = model(images) # take that whole batch in 
            loss = loss_function(outputs.squeeze(), labels) # ouptuts need to be squeezed to change shape from (B, 1) to just (B)
            # backward and optimize        
            # pros and cons of keeping the clip grad norm? working keeping?    
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            # get loss value and clean up
            train_loss += loss.item()

            del outputs, loss, images, labels # make sure to clean these up to keep the vram nice and neat :)

        
        # now for the validatino phase (we run these back to back for finetuning, less epochs)
        model.eval()
        val_loss = 0.0
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)

                outputs = model(images)
                loss = loss_function(outputs.squeeze(), labels)                
                val_loss += loss.item()

                preds = torch.sigmoid(outputs.squeeze())
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

                print(f"Val positives: {sum(all_labels)} / {len(all_labels)}")
                print(f"Prediction mean: {np.mean(all_preds):.4f}, std: {np.std(all_preds):.4f}")   

        val_auc = roc_auc_score(all_labels, all_preds)

        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)

        print(f"Epoch {epoch+1}/{num_epochs} | "
                f"Train Loss: {avg_train_loss:.4f} | "
                f"Val Loss: {avg_val_loss:.4f} | "
                f"Val AUC: {val_auc:.4f}")
        
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            torch.save(model.state_dict(), "Best_densenet_pneumonia.pth")
            print(f"New best model! AUC {best_val_auc:.4f}")

        history["train_loss"].append(avg_train_loss)
        history["val_loss"].append(avg_val_loss)
        history["val_auc"].append(val_auc)

    return history




In [164]:
# aight lets just see if this works

model = FinetunedDenseNet()
model = model.to(device)
# no longer need to do the weird he initialization, as I am using a pre-trained model. 

# lets start with the CheXNet init params cause why not
learning_rate = 0.0001 # decay by factor of 10 each time validation loss platueas after epoch
# lets also make it 10x smaller, 1e-4 should work better. 

subset = False # this can either make it smaller or bigger depending on our needs. 
# Adam optimizer, standard params. 
train_loader, test_loader, pos_weight = get_dataloaders(projections_path, reports_path, image_dir, subset=subset) # after that I got a bunch of defaults, I'll change em when I get there.

loss_function = nn.BCEWithLogitsLoss(pos_weight=pos_weight)  # trying a different loss function, we shall see.
optimizer = optim.AdamW(model.parameters(), lr=learning_rate) # 
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=3, factor=0.5)

history = train_model(model, train_loader, test_loader, 23, device, pos_weight)

Val positives: 19.0 / 128
Prediction mean: 0.5286, std: 0.0787
Val positives: 40.0 / 256
Prediction mean: 0.5309, std: 0.0802
Val positives: 64.0 / 384
Prediction mean: 0.5316, std: 0.0782
Val positives: 91.0 / 512
Prediction mean: 0.5328, std: 0.0774
Val positives: 113.0 / 640
Prediction mean: 0.5335, std: 0.0756
Val positives: 135.0 / 764
Prediction mean: 0.5349, std: 0.0762
Epoch 1/23 | Train Loss: 1.1579 | Val Loss: 1.1579 | Val AUC: 0.5034
New best model! AUC 0.5034
Val positives: 19.0 / 128
Prediction mean: 0.5793, std: 0.0775
Val positives: 40.0 / 256
Prediction mean: 0.5786, std: 0.0805
Val positives: 64.0 / 384
Prediction mean: 0.5783, std: 0.0811
Val positives: 91.0 / 512
Prediction mean: 0.5813, std: 0.0795
Val positives: 113.0 / 640
Prediction mean: 0.5827, std: 0.0775
Val positives: 135.0 / 764
Prediction mean: 0.5840, std: 0.0772
Epoch 2/23 | Train Loss: 1.1247 | Val Loss: 1.1502 | Val AUC: 0.5840
New best model! AUC 0.5840
Val positives: 19.0 / 128
Prediction mean: 0.532

In [165]:
df = build_master_df_for_pneumonia(projections_path, reports_path)
print(f"Total images: {len(df)}")
print(f"Unique filenames: {df['filename'].nunique()}")
print(f"Duplicate filenames: {len(df) - df['filename'].nunique()}")

Total images: 3818
Unique filenames: 3818
Duplicate filenames: 0


In [166]:
df = build_master_df_for_pneumonia(projections_path, reports_path)
print(f"DataFrame shape: {df.shape}")  # Should be (3818, 2)
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024:.1f} KB")  # Should be tiny

DataFrame shape: (3818, 2)
Memory usage: 306.0 KB
